<a href="https://colab.research.google.com/github/Rohitkumar6394/Text-summarizer/blob/main/Pdf_Summary_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers torch accelerate
!pip install -q pypdf
!pip install -q reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.0 MB/s eta 0:00:00


In [2]:
from google.colab import files

uploaded = files.upload()

Saving data.pdf to data.pdf


In [3]:
from pypdf import PdfReader

pdf_file = list(uploaded.keys())[0]

reader = PdfReader(pdf_file)

text = ""

for page in reader.pages:
    page_text = page.extract_text()

    if page_text:
        text += page_text + "\n"

print("Characters:", len(text))

Characters: 12667


In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM # Changed from pipeline to AutoTokenizer and AutoModelForSeq2SeqLM

class CustomSummarizerPipeline:
    def __init__(self, model_name):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    def __call__(self, texts, max_length=150, min_length=30, do_sample=False):
        if isinstance(texts, str):
            texts = [texts]

        results = []
        for text_to_summarize in texts:
            inputs = self.tokenizer(
                [text_to_summarize],
                max_length=1024,
                truncation=True,
                return_tensors="pt"
            )
            # Ensure model and inputs are on the same device (e.g., "cpu" or "cuda")
            # For simplicity, assuming CPU for now as device is not specified.
            # You might want to add .to(device) for GPU usage if available.
            summary_ids = self.model.generate(
                inputs["input_ids"],
                num_beams=4,
                max_length=max_length,
                min_length=min_length,
                early_stopping=True,
                do_sample=do_sample
            )
            results.append({"summary_text": self.tokenizer.decode(summary_ids[0], skip_special_tokens=True)})
        return results

model_name = "sshleifer/distilbart-cnn-12-6"
summarizer = CustomSummarizerPipeline(model_name) # Instantiate custom summarizer

print("Model Loaded (custom pipeline)") # Updated message

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

Model Loaded (custom pipeline)


In [6]:
def split_text(text, chunk_size=1000):

    chunks = []

    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i+chunk_size])

    return chunks

In [7]:
chunks = split_text(text)

summary_parts = []

for chunk in chunks:

    result = summarizer(
        chunk,
        max_length=150,
        min_length=40,
        do_sample=False
    )

    summary_parts.append(
        result[0]["summary_text"]
    )

final_summary = "\n\n".join(summary_parts)

print(final_summary)

 BSE CSE Project 2026: Health Management System v1.0.0 Guidelines Guidelines Guidelines . Guidelines: Guidelines, Guidelines, Constraints & Assumptions . Table of Contents: The requirements, specifications, requirements, requirements and requirements for the Health Management system .

 This SRS describes complete requirements for the Health Management System (HMS) v1.0 — a web-basedapplication designed to help hospitals and clinics manage patient information, doctor details, appointments, medical records, and billing efficiently . The HMS provides role-based portals for Admins, Doctors, and Patients . Excluded: Telemedicine video, third-party insurance gateway, native mobile ap .

 The HMS is a standalone web-based application using a threre-built application . The software is based on the Health Management System (HMS) v1.0.0 .

e-tier architecture: Frontend (Presentation) with HTML5, CSS3, JavaScript (ES6+) and Node.js v18+ with Express.js REST API .Database (Data) with MongoDB 6.x 

In [8]:
summary_file = "summary_memory.txt"

with open(summary_file, "w", encoding="utf-8") as f:
    f.write(final_summary)

print("Summary Saved")

Summary Saved


In [9]:
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer
)

from reportlab.lib.styles import getSampleStyleSheet

pdf = SimpleDocTemplate(
    "summary_output.pdf"
)

styles = getSampleStyleSheet()

content = []

content.append(
    Paragraph(
        "PDF Summary",
        styles["Title"]
    )
)

content.append(
    Spacer(1,12)
)

content.append(
    Paragraph(
        final_summary,
        styles["BodyText"]
    )
)

pdf.build(content)

print("PDF Generated")

PDF Generated


In [10]:
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer
)

from reportlab.lib.styles import getSampleStyleSheet

pdf = SimpleDocTemplate(
    "summary_output.pdf"
)

styles = getSampleStyleSheet()

content = []

content.append(
    Paragraph(
        "PDF Summary",
        styles["Title"]
    )
)

content.append(
    Spacer(1,12)
)

content.append(
    Paragraph(
        final_summary,
        styles["BodyText"]
    )
)

pdf.build(content)

print("PDF Generated")

PDF Generated


In [11]:
from google.colab import files

files.download("summary_output.pdf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
with open(
    "summary_memory.txt",
    "a",
    encoding="utf-8"
) as f:

    f.write("\n")
    f.write("="*50)
    f.write("\n")

    f.write(final_summary)
    f.write("\n")